# 06b - Prepare External Fine-tuning Dataset

Converts the instructor-shared `data/external/llm.jsonl` SFT examples into the same instruction format, applies benchmark leakage filtering, and creates a combined fine-tuning dataset. This notebook still does not train the model.

In [ ]:
from pathlib import Path
import json
import sys
import importlib

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path = [str(DRIVE_ROOT)] + [p for p in sys.path if p != str(DRIVE_ROOT)]
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

config = json.loads((DRIVE_ROOT / 'project_config.json').read_text(encoding='utf-8'))
external_llm = DRIVE_ROOT / config.get('external_dataset', {}).get('llm_sft', 'data/external/llm.jsonl')
benchmark_csv = DRIVE_ROOT / config['benchmark_csv']
existing_train = DRIVE_ROOT / 'data/processed/finetune_train.jsonl'
existing_val = DRIVE_ROOT / 'data/processed/finetune_val.jsonl'

paths = [external_llm, benchmark_csv, existing_train, existing_val]
for path in paths:
    if not path.exists():
        raise FileNotFoundError(path)

external_llm, benchmark_csv, existing_train, existing_val

In [ ]:
from src.prepare_external_finetune_dataset import prepare_external_and_combined_finetune_dataset

report = prepare_external_and_combined_finetune_dataset(
    external_llm_jsonl=external_llm,
    benchmark_csv=benchmark_csv,
    existing_train_jsonl=existing_train,
    existing_val_jsonl=existing_val,
    output_train_jsonl=DRIVE_ROOT / 'data/processed/finetune_train_combined.jsonl',
    output_val_jsonl=DRIVE_ROOT / 'data/processed/finetune_val_combined.jsonl',
    output_external_train_jsonl=DRIVE_ROOT / 'data/processed/finetune_external_train.jsonl',
    output_external_val_jsonl=DRIVE_ROOT / 'data/processed/finetune_external_val.jsonl',
    report_json=DRIVE_ROOT / 'reports/leakage_report_external.json',
    val_ratio=0.1,
    seed=42,
    similarity_threshold=0.88,
    max_external_samples=None,
)

report

In [ ]:
import json

for rel in [
    'data/processed/finetune_external_train.jsonl',
    'data/processed/finetune_external_val.jsonl',
    'data/processed/finetune_train_combined.jsonl',
    'data/processed/finetune_val_combined.jsonl',
]:
    path = DRIVE_ROOT / rel
    print(path, path.exists(), round(path.stat().st_size / 1024, 1), 'KB')
    with path.open('r', encoding='utf-8') as f:
        sample = json.loads(next(f))
    print(sample.keys())
    print(sample['instruction'][:160])
    print(sample['input'][:220])
    print(sample['output'][:220])
    print('---')

Expected outputs:

- `data/processed/finetune_external_train.jsonl`
- `data/processed/finetune_external_val.jsonl`
- `data/processed/finetune_train_combined.jsonl`
- `data/processed/finetune_val_combined.jsonl`
- `reports/leakage_report_external.json`

Recommended training input for LoRA/QLoRA: the combined files, because they include both project QA auxiliary data and source-grounded external SFT examples.